# 제9장. 베이지안 의사결정
## 3시간 인터랙티브 강의 (이론 중심 + 실습)

**학습 목표:**
1. 점 추정과 확률 분포의 차이를 **직관적으로** 이해
2. 베이즈 정리의 **작동 원리**를 수학적·시각적으로 이해
3. 순차적 베이지안 업데이트의 **의미**를 체득
4. 정보의 가치(EVPI, EVII)를 **의사결정**에 활용

**강의 구조:**
- **Part 1**: 📖 점 추정의 문제점 (이론 + 과제)
- **Part 2**: 📖 베이즈 정리 (이론 + 과제 + 실습)
- *휴식*
- **Part 3**: 📖 베이지안 업데이트 (이론 + 과제)
- **Part 4**: 🔬 실습 - 순차적 업데이트
- **Part 5**: 🔬 실습 - EVPI/EVII 계산

---

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly", "scipy", "matplotlib"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Matplotlib 한글 폰트 설정 (크로스 플랫폼)
import platform
available_fonts = [f.name for f in fm.fontManager.ttflist]
if platform.system() == 'Darwin' and 'AppleGothic' in available_fonts:
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows' and 'Malgun Gothic' in available_fonts:
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

---

## Part 1: 📖 점 추정의 문제점

### 1.1 동기부여: 왜 확률적 사고가 필요한가?

#### 시나리오

당신은 신규 모바일 앱 출시를 기획하는 PM입니다. 시장조사 보고서에는 다음과 같이 적혀 있습니다:

> **"목표 시장 규모는 100억 원으로 추정됩니다."**

CEO가 묻습니다:
- "100억이면 투자할 만한가요? 우리 손익분기점은 80억입니다."
- 당신의 답변은?

#### 문제점 1: 단일 숫자의 함정

"100억"이라는 **점 추정(Point Estimate)**은 다음을 감춥니다:

1. **오차 범위**: 실제로 80~120억인지, 50~200억인지 모름
2. **확실성 정도**: 90% 확신인지, 50% 확신인지 모름
3. **위험**: 손실 가능성이 얼마나 되는지 모름

#### 문제점 2: 과신(Overconfidence)

인간의 뇌는 불확실성을 싫어합니다. 단일 숫자를 보면 마치 그것이 **확실한 것처럼** 느껴집니다.

**Kahneman (2011)** 연구:
- 전문가들도 자신의 추정치에 지나친 확신을 가짐
- "좁은 프레이밍(Narrow Framing)" 문제
- 실제 정확도: 60~70%, 본인 생각: 90% 이상

---

### 📝 이론 과제 1-1 (15분)

#### 과제: 점 추정의 위험성 조사

**질문:**
1. "점 추정(Point Estimate)"이란 무엇인가요? (2-3문장으로 설명)
2. 점 추정이 의사결정에 미치는 **부정적 영향** 3가지를 찾아서 설명하세요.
3. 실제 사례를 1개 찾아서 요약하세요. (예: 프로젝트 실패 사례, 투자 손실 등)

**조사 방법:**
- 구글 검색: "point estimate risk", "overconfidence bias"
- 위키피디아, 학술 블로그, 뉴스 기사 등 활용

**제출 형식:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

---

### ✅ 과제 1-1 제출란

**학번:** ___________  
**이름:** ___________

#### 1. 점 추정이란?

(여기에 작성)

#### 2. 점 추정의 부정적 영향 3가지

1. 
2. 
3. 

#### 3. 실제 사례

(사례 요약)

**출처:** (URL 또는 문헌)

---

### 1.2 해결책: 확률 분포로 불확실성 표현

#### 개념

**점 추정 대신:**
- "시장 규모는 100억 원입니다." ❌

**확률 분포로:**
- "시장 규모는 **평균 100억, 표준편차 30억**의 정규분포를 따릅니다." ✅
- "90% 신뢰구간: 51억 ~ 149억 원"
- "80억 초과 확률: 75%"

#### 왜 더 나은가?

1. **불확실성 명시**: "얼마나 확실한지" 정량적으로 표현
2. **위험 평가**: 손실 확률을 계산 가능
3. **의사결정 개선**: "75% 확률"이라는 정보로 더 나은 판단

---

### 1.3 시각화: 점 추정 vs 확률 분포

#### 시뮬레이션

시장 규모를 10,000번 시뮬레이션합니다.
- 평균: 100억 원
- 표준편차: 30억 원
- 투자 기준: 80억 원

In [ ]:
# 시장 규모 시뮬레이션 (10,000회)
market_sizes = np.random.normal(100, 30, 10000)  # 평균 100억, 표준편차 30억

# 투자 기준 (80억) 초과 확률
prob_success = (market_sizes > 80).sum() / len(market_sizes)

# 통계
mean = market_sizes.mean()
std = market_sizes.std()
ci_5 = np.percentile(market_sizes, 5)
ci_95 = np.percentile(market_sizes, 95)

print("📊 시장 규모 시뮬레이션 결과 (10,000회):")
print(f"  - 평균: {mean:.1f}억 원")
print(f"  - 표준편차: {std:.1f}억 원")
print(f"  - 90% 신뢰구간: [{ci_5:.1f}억, {ci_95:.1f}억]")
print(f"\n  ✅ 투자 기준(80억) 초과 확률: {prob_success:.1%}")
print(f"  ⚠️  손실 위험(80억 미만 확률): {1-prob_success:.1%}")
print(f"\n💡 해석: '100억'이라는 단일 숫자보다 '{prob_success:.0%} 확률로 80억 초과'가 정확!")

In [ ]:
# Plotly 시각화
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=market_sizes,
    nbinsx=50,
    name='시장 규모 분포',
    marker_color='lightblue',
    opacity=0.7
))

fig.add_vline(x=80, line_dash='dash', line_color='red', line_width=3,
              annotation_text=f'투자 기준 (80억)<br>초과 확률: {prob_success:.1%}',
              annotation_position='top right')

fig.add_vline(x=100, line_dash='solid', line_color='green', line_width=3,
              annotation_text='평균 (100억)',
              annotation_position='top left')

# 손실 영역 표시
fig.add_vrect(x0=0, x1=80, fillcolor='red', opacity=0.1, 
              annotation_text=f"손실 위험<br>{1-prob_success:.1%}",
              annotation_position="top left")

fig.update_layout(
    title='시장 규모 확률 분포 (10,000회 시뮬레이션)',
    xaxis_title='시장 규모 (억 원)',
    yaxis_title='빈도',
    height=500,
    showlegend=False
)

fig.show()

#### 토론 질문 (5분)

1. CEO에게 "시장 규모는 100억 원입니다"라고 보고하는 것과  
   "75% 확률로 80억을 초과하지만, 25% 위험이 있습니다"라고 보고하는 것,  
   어느 쪽이 더 정직하고 유용한가요?

2. 만약 손실 위험이 40%라면 투자 결정이 바뀌어야 할까요?

3. 불확실성을 명시하면 CEO가 "너무 불확실하다"며 화낼까요?  
   아니면 "정직하다"며 신뢰할까요?

---

---

## Part 2: 📖 베이즈 정리

### 2.1 베이즈 정리란 무엇인가?

#### 일상적 정의

베이즈 정리(Bayes' Theorem)는 **"새로운 증거를 보고 믿음을 어떻게 업데이트해야 하는지"**를 알려주는 수학 공식입니다.

#### 직관적 예시

**상황:** 친구가 오늘 지각했습니다.

**사전 믿음 (Prior):**
- 친구는 평소 성실해서 지각 확률 10%

**새로운 증거 (Evidence):**
- 오늘 비가 많이 왔음
- 비 오는 날 지각 확률: 50%
- 비 안 오는 날 지각 확률: 5%

**사후 믿음 (Posterior):**
- 비가 왔다는 증거를 보고 나니, 친구 지각 확률을 어떻게 업데이트해야 할까요?

→ 베이즈 정리가 정확한 답을 줍니다!

---

### 2.2 베이즈 정리의 수학적 형태

#### 공식

$$P(H|D) = \frac{P(D|H) \times P(H)}{P(D)}$$

#### 각 항의 의미

| 기호 | 이름 | 의미 | 예시 |
|------|------|------|------|
| $P(H)$ | **사전확률** (Prior) | 증거 보기 **전** 가설에 대한 믿음 | 친구 평소 지각률 10% |
| $P(D\|H)$ | **가능도** (Likelihood) | 가설이 참일 때 증거가 나올 확률 | 친구가 지각하면 비가 왔을 확률 |
| $P(H\|D)$ | **사후확률** (Posterior) | 증거 본 **후** 업데이트된 믿음 | 비가 왔다는 걸 알고 나서 지각 확률 |
| $P(D)$ | **증거** (Evidence) | 증거가 발생할 전체 확률 | 비가 올 확률 (normalizing constant) |

#### 직관적 이해

```
사후확률 = (가능도 × 사전확률) / 증거

↓ 말로 풀면

"새 증거를 본 후 믿음" = "증거가 가설과 얼마나 일치하는지" × "원래 믿음" / "증거의 흔함"
```

---

### 📝 이론 과제 2-1 (15분)

#### 과제: 베이즈 정리 심화 이해

**질문:**

1. **사전확률(Prior)**의 역할은 무엇인가요? 왜 중요한가요? (3-4문장)

2. **가능도(Likelihood)**와 **사후확률(Posterior)**의 차이를 설명하세요. (3-4문장)

3. "기저율 무시 오류(Base Rate Neglect)"란 무엇인가요? 실제 사례를 찾아서 요약하세요.

**조사 키워드:**
- "Bayes theorem intuition"
- "prior probability importance"
- "base rate neglect"
- "likelihood vs posterior"

**제출:** 아래 마크다운 셀에 **직접 타이핑**

---

### ✅ 과제 2-1 제출란

**학번:** ___________  
**이름:** ___________

#### 1. 사전확률의 역할

(여기에 작성)

#### 2. 가능도 vs 사후확률

(여기에 작성)

#### 3. 기저율 무시 오류 사례

(사례 요약)

**출처:** (URL)

---

### 2.3 의료 진단 예시: 베이즈 정리의 위력

#### 시나리오

희귀 질병 X의 검사 결과가 **양성**으로 나왔습니다.

**주어진 정보:**
- **유병률(Base Rate)**: 1% (100명 중 1명)
- **민감도(Sensitivity)**: 99% (실제 환자를 양성으로 판정)
- **특이도(Specificity)**: 95% (건강한 사람을 음성으로 판정)

**질문:** 양성 판정을 받았을 때, 실제로 이 질병에 걸렸을 확률은?

#### 대부분 사람의 직관적 답변

"검사가 99% 정확하니까... 거의 99%겠지?"

#### 실제 답 (베이즈 정리)

**약 16.7%**

→ 왜 이렇게 낮을까요? **기저율(유병률 1%)**이 핵심입니다!

---

### 2.4 단계별 계산 (10,000명 시뮬레이션)

#### 이해를 돕는 구체적 숫자

In [ ]:
# 주어진 정보
유병률 = 0.01       # 1%
민감도 = 0.99       # 99%
특이도 = 0.95       # 95%
총인구 = 10000

# 1단계: 실제 환자 수와 건강한 사람 수
환자수 = int(총인구 * 유병률)
건강인수 = 총인구 - 환자수

print("[1단계] 10,000명 중:")
print(f"  - 실제 환자: {환자수}명")
print(f"  - 건강한 사람: {건강인수}명")
print()

# 2단계: 진양성(TP)과 위양성(FP)
진양성 = int(환자수 * 민감도)        # 환자 중 양성 판정
위양성 = int(건강인수 * (1 - 특이도))  # 건강한 사람 중 양성 판정

print("[2단계] 검사 결과:")
print(f"  - 진양성 (TP): {진양성}명 (실제 환자 → 양성 판정)")
print(f"  - 위양성 (FP): {위양성}명 (건강한 사람 → 양성 판정)")
print()

# 3단계: 총 양성 판정자
총양성 = 진양성 + 위양성

print("[3단계] 양성 판정 받은 사람:")
print(f"  - 총 {총양성}명 (진양성 {진양성}명 + 위양성 {위양성}명)")
print()

# 4단계: 베이즈 정리 - 양성 중 실제 환자 확률
양성중환자확률 = 진양성 / 총양성

print("[4단계] 베이즈 정리 적용:")
print(f"  P(환자|양성) = 진양성 / 총양성")
print(f"               = {진양성} / {총양성}")
print(f"               = {양성중환자확률:.3f}")
print()
print(f"✅ 답: 양성 판정 시 실제 환자일 확률은 {양성중환자확률:.1%}")
print()
print("💡 핵심 통찰:")
print(f"   - 검사가 99% 정확해도, 유병률이 1%로 낮으면")
print(f"   - 양성 판정의 {위양성}/{총양성} = {위양성/총양성:.1%}가 위양성!")
print(f"   - 기저율(Base Rate)을 무시하면 안 됩니다!")

#### 시각화: 혼동 행렬 (Confusion Matrix)

In [ ]:
# 혼동 행렬 데이터
import plotly.figure_factory as ff

# 2x2 테이블
z = [[진양성, 환자수 - 진양성],
     [위양성, 건강인수 - 위양성]]

x = ['양성 판정', '음성 판정']
y = ['실제 환자', '실제 건강']

z_text = [[f'{진양성}명<br>(TP)', f'{환자수 - 진양성}명<br>(FN)'],
          [f'{위양성}명<br>(FP)', f'{건강인수 - 위양성}명<br>(TN)']]

fig = ff.create_annotated_heatmap(z, x=x, y=y, annotation_text=z_text,
                                   colorscale='Blues', showscale=False)

fig.update_layout(
    title=f'혼동 행렬 (10,000명 기준)<br>양성 {총양성}명 중 실제 환자는 {진양성}명 ({양성중환자확률:.1%})',
    xaxis_title='검사 결과',
    yaxis_title='실제 상태',
    height=400
)

fig.show()

### 2.5 유병률 변화에 따른 양성예측도

#### 질문

유병률이 높아지면 양성 판정의 신뢰도가 어떻게 변할까요?

In [ ]:
# 유병률을 1%에서 50%까지 변화
prevalence_range = np.linspace(0.01, 0.5, 50)
ppv_list = []

for prev in prevalence_range:
    tp = prev * 민감도
    fp = (1 - prev) * (1 - 특이도)
    ppv = tp / (tp + fp)
    ppv_list.append(ppv)

# Plotly 그래프
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prevalence_range * 100,
    y=np.array(ppv_list) * 100,
    mode='lines',
    line=dict(color='blue', width=3),
    name='양성예측도 (PPV)',
    fill='tozeroy',
    fillcolor='rgba(0,0,255,0.1)'
))

fig.add_hline(y=50, line_dash='dash', line_color='red', line_width=2,
              annotation_text='50% 기준선')

# 유병률 1% 지점 표시
fig.add_vline(x=1, line_dash='dot', line_color='gray',
              annotation_text=f'유병률 1%<br>PPV: {양성중환자확률:.1%}')

fig.update_layout(
    title='유병률에 따른 양성예측도 변화 (민감도 99%, 특이도 95%)',
    xaxis_title='유병률 (%)',
    yaxis_title='양성 시 실제 환자 확률 (%)',
    height=500
)

fig.show()

print("\n📌 핵심 통찰:")
print("  - 유병률이 낮으면(< 5%) 양성의 대부분이 위양성")
print("  - 유병률 약 5%가 되어야 PPV가 50%를 넘음")
print("  - 검사 정확도보다 기저율(Base Rate)이 더 중요!")

### 📝 실습 과제 2-2 (10분)

#### 과제: 베이즈 정리 손계산

**시나리오:**

회사 보안 시스템이 직원의 부정행위를 탐지합니다.
- **실제 부정행위 비율**: 2% (100명 중 2명)
- **탐지 정확도 (민감도)**: 90% (부정행위자를 경고로 표시)
- **특이도**: 85% (정직한 직원을 안전으로 표시)

**질문:**

1. 1,000명 직원 중 실제 부정행위자는 몇 명인가요?
2. 시스템이 **경고**를 표시한 사람은 총 몇 명인가요? (진양성 + 위양성)
3. 경고를 받은 사람이 **실제로 부정행위자일 확률**은?

**힌트:** 위 의료 진단 예시를 참고하세요.

---

### ✅ 과제 2-2 제출란

**학번:** ___________  
**이름:** ___________

#### 계산 과정 (손으로 계산 후 타이핑)

**1. 실제 부정행위자 수:**

(계산 과정)

답: _____명

**2. 경고 받은 총 인원:**

- 진양성(TP): (계산)
- 위양성(FP): (계산)
- 총 경고: (계산)

답: _____명

**3. 경고 받은 사람이 실제 부정행위자일 확률:**

(베이즈 정리 적용)

답: _____% (약 _____%)

---

---

## 🕐 휴식 시간

15분 휴식 후 **Part 3**에서 계속됩니다.

---

## Part 3: 📖 베이지안 업데이트

### 3.1 순차적 업데이트: 학습하는 AI처럼

#### 개념

베이지안의 **핵심 강점**: 새로운 정보가 들어올 때마다 믿음을 **갱신(Update)**할 수 있다!

**공식:**
```
오늘의 사후분포 = 내일의 사전분포
```

#### 일상 비유

**상황:** 새로 만난 친구가 약속을 지킬지 판단

1. **Day 1** (사전): "사람은 대체로 약속을 지킨다" (70%)
2. **Day 2** (증거 1): 첫 약속을 지켰다 → 믿음 상승 (85%)
3. **Day 3** (증거 2): 두 번째도 지켰다 → 믿음 더 상승 (92%)
4. **Day 4** (증거 3): 세 번째도 지켰다 → 확신 (96%)

데이터가 쌓일수록 **불확실성 감소**, **확신 증가**!

---

### 3.2 Beta 분포: 성공/실패 확률 추정

#### 왜 Beta 분포인가?

성공률(확률 0~1)을 추정할 때 **수학적으로 편리**한 분포입니다.

**Beta 분포의 특성:**
- **Beta(α, β)**: α = 성공 수 + 1, β = 실패 수 + 1
- **평균**: α / (α + β)
- **업데이트**: 데이터 추가 시 α, β만 업데이트하면 됨!

#### 예시: 신제품 구매 전환율

**사전 믿음:**
- "대략 20% 정도일 것" → Beta(2, 8)
- 평균 = 2/(2+8) = 20%

**데이터 관측:**
- 10명 중 3명 구매 (성공 3, 실패 7)

**사후분포:**
- Beta(2+3, 8+7) = Beta(5, 15)
- 평균 = 5/(5+15) = 25%

---

### 📝 이론 과제 3-1 (15분)

#### 과제: 베이지안 업데이트 심화

**질문:**

1. "순차적 업데이트(Sequential Update)"가 왜 중요한가요? (3-4문장)

2. 데이터가 많아질수록 **사전확률의 영향**은 어떻게 변하나요? (3-4문장)

3. Beta 분포가 "켤레 사전분포(Conjugate Prior)"라고 불리는 이유를 조사하세요.

**조사 키워드:**
- "sequential Bayesian updating"
- "prior converges to posterior"
- "beta distribution conjugate prior"

---

### ✅ 과제 3-1 제출란

**학번:** ___________  
**이름:** ___________

#### 1. 순차적 업데이트의 중요성

(여기에 작성)

#### 2. 사전확률의 영향 변화

(여기에 작성)

#### 3. 켤레 사전분포란?

(조사 내용)

**출처:** (URL)

---

### 3.3 사전분포의 수렴: 결국 데이터가 이긴다

#### 실험

**4가지 극단적인 사전분포**에서 시작해도, 충분한 데이터가 있으면 **같은 결론**으로 수렴!

1. **무정보 사전**: Beta(1, 1) - 아무 정보 없음
2. **비관적 사전**: Beta(2, 18) - 전환율 10% 믿음
3. **중립 사전**: Beta(5, 5) - 전환율 50% 믿음
4. **낙관적 사전**: Beta(18, 2) - 전환율 90% 믿음

**동일 데이터:** 200명 중 68명 성공 (실제 전환율 34%)

In [ ]:
# 4가지 사전분포
priors = {
    "무정보 (1%)": (1, 1),
    "비관적 (10%)": (2, 18),
    "중립 (50%)": (5, 5),
    "낙관적 (90%)": (18, 2)
}

# 동일 데이터
successes = 68
failures = 200 - 68

print("📊 사전분포의 수렴 실험")
print("=" * 60)
print(f"동일 데이터: 200명 중 {successes}명 성공 (전환율 {successes/200:.1%})")
print()

fig = go.Figure()
x = np.linspace(0, 1, 1000)

# 사전분포별 시각화
colors = ['gray', 'red', 'blue', 'green']
for i, (name, (alpha, beta)) in enumerate(priors.items()):
    # 사전분포
    prior_mean = alpha / (alpha + beta)
    prior_pdf = stats.beta.pdf(x, alpha, beta)
    
    fig.add_trace(go.Scatter(
        x=x*100, y=prior_pdf,
        mode='lines',
        name=f'{name} 사전',
        line=dict(color=colors[i], width=1, dash='dash'),
        showlegend=True
    ))
    
    # 사후분포
    post_alpha = alpha + successes
    post_beta = beta + failures
    post_mean = post_alpha / (post_alpha + post_beta)
    post_pdf = stats.beta.pdf(x, post_alpha, post_beta)
    
    fig.add_trace(go.Scatter(
        x=x*100, y=post_pdf,
        mode='lines',
        name=f'{name} 사후',
        line=dict(color=colors[i], width=3),
        showlegend=True
    ))
    
    print(f"{name}")
    print(f"  사전평균: {prior_mean:.1%} → 사후평균: {post_mean:.1%}")
    print()

# 실제 전환율
fig.add_vline(x=34, line_dash='dot', line_color='black', line_width=2,
              annotation_text='실제 전환율 (34%)')

fig.update_layout(
    title='사전분포의 수렴: 데이터가 많으면 사전분포는 중요하지 않다',
    xaxis_title='전환율 (%)',
    yaxis_title='확률 밀도',
    height=500
)

fig.show()

print("\n💡 핵심 통찰:")
print("  - 사전분포가 10%~90%로 극단적으로 달랐지만")
print("  - 200개 데이터 후 모두 32%~36% 범위로 수렴")
print("  - 충분한 데이터가 있으면 사전분포 선택은 덜 중요!")

---

## Part 4: 🔬 실습 - 순차적 업데이트

### 실습 배경

여러분은 신규 온라인 서비스의 **일일 가입자 수**를 추정합니다.

**사전 믿음:** 평균 100명, 표준편차 30명

**첫 주 데이터 (월~일):** 85, 112, 95, 128, 103, 91, 110명

**측정 불확실성:** 표준편차 20명

---

### 📝 실습 과제 4-1 (25분)

#### 과제: 정규-정규 켤레 업데이트 구현

정규 분포끼리의 베이지안 업데이트를 직접 구현하세요.

**공식:**

$$\text{사후 정밀도} = \text{사전 정밀도} + \text{데이터 정밀도}$$

$$\text{사후 평균} = \frac{\text{사전 정밀도} \times \text{사전 평균} + \text{데이터 정밀도} \times \text{데이터 평균}}{\text{사후 정밀도}}$$

**정밀도 = 1 / 분산**

---

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# 주어진 정보
prior_mean = 100  # 사전 평균
prior_std = 30    # 사전 표준편차
data_std = 20     # 측정 불확실성

# 관측 데이터
data = np.array([85, 112, 95, 128, 103, 91, 110])
n = len(data)
data_mean = data.mean()

# TODO: 1. 정밀도 계산 (precision = 1 / variance)
prior_precision = # 여기에 코드 작성
data_precision = # 여기에 코드 작성

# TODO: 2. 사후분포 계산
post_precision = # 여기에 코드 작성
post_mean = # 여기에 코드 작성 (위 공식 참고)
post_std = 1 / np.sqrt(post_precision)

# ========== 여기까지 ==========

# 결과 출력
print("\n📊 베이지안 업데이트 결과:")
print(f"\n[사전분포]")
print(f"  - 평균: {prior_mean:.1f}명")
print(f"  - 표준편차: {prior_std:.1f}명")

print(f"\n[관측 데이터]")
print(f"  - 샘플 수: {n}일")
print(f"  - 데이터 평균: {data_mean:.1f}명")

print(f"\n[사후분포]")
print(f"  - 평균: {post_mean:.1f}명")
print(f"  - 표준편차: {post_std:.1f}명")

print(f"\n💡 해석:")
print(f"  - 평균이 {prior_mean}명 → {post_mean:.1f}명")
print(f"  - 불확실성 {prior_std}명 → {post_std:.1f}명으로 감소")

### ✅ 실습 과제 4-1 제출란

**학번:** ___________  
**이름:** ___________

**질문 1:** 사후평균은?

답: 약 _______명

**질문 2:** 사후 표준편차는?

답: 약 _______명

**질문 3:** 사전평균(100명)과 데이터 평균 중 어느 쪽에 더 가깝나요? 왜 그럴까요?

답: (설명)

---

---

## Part 5: 🔬 실습 - EVPI/EVII 계산

### 실습 배경

기업 A는 신규 시장 진출을 검토합니다.

**의사결정 상황:**
- 투자 비용: 50억 원
- 시장 큼: 수익 100억 (순이익 50억)
- 시장 작음: 수익 20억 (순손실 30억)
- 시장이 클 확률: 40%

**시장조사 옵션:**
- 비용: 10억 원
- 정확도: 80%

**질문: 조사를 실시해야 하는가?**

---

### 📝 실습 과제 5-1 (25분)

#### 과제: EVPI와 EVII 계산

단계별로 계산하세요.

---

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# 주어진 정보
투자비용 = 50
수익_큼 = 100
수익_작음 = 20
이익_큼 = 수익_큼 - 투자비용  # 50억
이익_작음 = 수익_작음 - 투자비용  # -30억

사전확률_큼 = 0.40
사전확률_작음 = 1 - 사전확률_큼

조사비용 = 10
조사정확도 = 0.80

# TODO: 1. 현재 최적 결정
기대이익_진출 = # 여기에 코드 작성
기대이익_미진출 = 0

현재최적결정 = "진출" if 기대이익_진출 > 기대이익_미진출 else "미진출"
현재기대이익 = max(기대이익_진출, 기대이익_미진출)

print("[1단계] 현재 최적 결정")
print(f"  진출 기대이익: {기대이익_진출:.1f}억")
print(f"  미진출 기대이익: {기대이익_미진출:.1f}억")
print(f"  ✅ 최적: {현재최적결정} ({현재기대이익:.1f}억)")
print()

# TODO: 2. EVPI 계산
완전정보하_기대이익 = # 여기에 코드 작성 (힌트: 시장 크면 진출, 작으면 미진출)
EVPI = # 여기에 코드 작성

print("[2단계] 완전 정보의 기대가치 (EVPI)")
print(f"  완전 정보 하 기대이익: {완전정보하_기대이익:.1f}억")
print(f"  ✅ EVPI = {EVPI:.1f}억")
print()

# TODO: 3. EVII 계산 (정확도 80%)
# 베이즈 정리로 사후확률 계산
prob_결과크다 = # 여기에 코드 작성
prob_결과작다 = 1 - prob_결과크다

사후확률_큼_결과크다 = # 베이즈 정리 적용
사후확률_큼_결과작다 = # 베이즈 정리 적용

# 조사 결과별 최적 결정
기대이익_진출_결과크다 = # 여기에 코드 작성
최적기대이익_결과크다 = max(기대이익_진출_결과크다, 0)

기대이익_진출_결과작다 = # 여기에 코드 작성
최적기대이익_결과작다 = max(기대이익_진출_결과작다, 0)

조사후_기대이익 = # 여기에 코드 작성
EVII = # 여기에 코드 작성

print("[3단계] 불완전 정보의 기대가치 (EVII)")
print(f"  조사 후 기대이익: {조사후_기대이익:.1f}억")
print(f"  ✅ EVII = {EVII:.1f}억")
print()

# TODO: 4. 조사 실시 여부
print("[4단계] 조사 실시 여부 결정")
print(f"  조사 비용: {조사비용}억")
print(f"  조사 가치 (EVII): {EVII:.1f}억")
print(f"  순이익: {EVII - 조사비용:.1f}억")

if EVII > 조사비용:
    print(f"  ✅ 조사 실시 권고 (순이익 {EVII - 조사비용:.1f}억)")
else:
    print(f"  ❌ 조사 불필요 (손실 {조사비용 - EVII:.1f}억)")

# ========== 여기까지 ==========

### ✅ 실습 과제 5-1 제출란

**학번:** ___________  
**이름:** ___________

**질문 1:** 현재 최적 결정은?

답: □ 진출 □ 미진출 (기대이익: _______억)

**질문 2:** EVPI는?

답: _______억

**질문 3:** EVII는?

답: _______억

**질문 4:** 조사를 실시해야 하나요?

답: □ 예 □ 아니오 (순이익: _______억)

**질문 5:** 만약 조사 비용이 15억이라면 결정이 바뀌나요? 왜 그런가요?

답: (설명)

---

---

## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

#### 1. 점 추정의 문제점
- 불확실성을 숨기고 과신 유발
- 확률 분포로 표현하면 더 정직하고 유용

#### 2. 베이즈 정리
- **사전확률 + 가능도 → 사후확률**
- 기저율(Base Rate)을 무시하면 안 됨
- 의료 진단 예시: 99% 정확해도 양성의 대부분이 위양성

#### 3. 베이지안 업데이트
- 새 데이터로 믿음을 순차적으로 갱신
- 데이터가 많아질수록 불확실성 감소
- 충분한 데이터가 있으면 사전분포는 덜 중요

#### 4. 정보의 가치
- **EVPI**: 완전 정보의 최대 가치
- **EVII**: 실제 조사의 가치
- 조사 비용 < EVII → 조사 실시

---

### 핵심 메시지

> **"불확실성을 숨기지 말고 드러내라."**
>
> 점 추정은 편리하지만 거짓된 확신을 준다.  
> 베이지안 사고는 "모른다"를 인정하면서도 최선의 결정을 내리는 방법이다.  
> 새로운 정보가 오면 믿음을 업데이트하라. 그것이 학습하는 기획자의 자세이다.

---

### 다음 장 예고

**제10장: 시나리오 플래닝**
- 여러 불확실성이 결합하여 다양한 미래를 만들 때
- 베이지안으로 추정한 변수들을 시나리오로 조합

**제11장: 몬테카를로 시뮬레이션**
- 확률 분포에서 무작위 샘플링
- "성공 확률 60%"가 구체적으로 무엇을 의미하는지

---

## 📚 추가 학습 자료

### 참고문헌
- McElreath, R. (2020). *Statistical Rethinking* (2nd ed.). CRC Press.
- Gelman, A., et al. (2013). *Bayesian Data Analysis* (3rd ed.). CRC Press.
- Kahneman, D. (2011). *Thinking, Fast and Slow*. Farrar, Straus and Giroux.

### Python 라이브러리
- `scipy.stats`: 통계 분포와 계산
- `PyMC`: 확률적 프로그래밍
- `ArviZ`: 베이지안 시각화

---

**수고하셨습니다! 👏**